# Game Playing as Adversarial Search

We implement core game-playing algorithms from scratch:
1. Minimax
2. Alpha-Beta Pruning
3. Depth-Limited Search with Evaluation Functions
4. Expectiminimax

We use small tree-based games to understand the mathematical ideas behind adversarial search.

## Learning Objectives

By the end of this notebook, you should be able to:

- represent a game tree computationally
- distinguish MAX nodes, MIN nodes, terminal nodes, and chance nodes
- implement minimax recursively
- understand why greedy search fails in adversarial settings
- implement alpha-beta pruning and observe pruning behavior
- use cutoff depth and evaluation functions for approximate search
- implement expectiminimax for stochastic games
- compare the algorithms conceptually and computationally

In [5]:
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Tuple
import math

---
## 1. Representing a Game Tree

We start with a small abstract game tree.

Each node may be one of:
- MAX
- MIN
- CHANCE
- TERMINAL

For terminal nodes, we store a utility value.
For chance nodes, we also store probabilities on outgoing edges.

In [16]:
@dataclass
class GameNode:
    name: str
    node_type: str   # "MAX", "MIN", "CHANCE", "TERMINAL"
    children: List["GameNode"] = field(default_factory=list)
    utility: Optional[float] = None
    probabilities: Optional[List[float]] = None  # only for CHANCE nodes

A terminal node has a utility value from MAX's perspective:
- large positive value => good for MAX
- large negative value => good for MIN

---
## 2. Example Tree for Minimax
We first construct a deterministic game tree:
```text
                 A (MAX)
               /         \
           B (MIN)      C (MIN)
          /   |   \      /   |   \
         3   12   8     2    4    6
```
Minimax reasoning:
- `B` chooses `min(3, 12, 8) = 3`
- `C` chooses `min(2, 4, 6) = 2`
- `A` chooses `max(3, 2) = 3`

In [18]:
# Leaves
n3 = GameNode("3", "TERMINAL", utility=3)
n12 = GameNode("12", "TERMINAL", utility=12)
n8 = GameNode("8", "TERMINAL", utility=8)
n2 = GameNode("2", "TERMINAL", utility=2)
n4 = GameNode("4", "TERMINAL", utility=4)
n6 = GameNode("6", "TERMINAL", utility=6)

# Internal nodes
B = GameNode("B", "MIN", children=[n3, n12, n8])
C = GameNode("C", "MIN", children=[n2, n4, n6])

# Root
A = GameNode("A", "MAX", children=[B, C])

---
## 3. Helper Function to Print a Tree

In [19]:
def print_tree(node: GameNode, depth: int=0):
    indent = "    " * depth
    if node.node_type == "TERMINAL":
        print(f"{indent}{node.name} [{node.node_type}, utility={node.utility}]")
    elif node.node_type == "CHANCE":
        print(f"{indent}{node.name} [{node.node_type}, probs={node.probabilities}]")
        for child in node.children:
            print_tree(child, depth + 1)
    else:
        print(f"{indent}{node.name} [{node.node_type}]")
        for child in node.children:
            print_tree(child, depth + 1)

print_tree(A)

A [MAX]
    B [MIN]
        3 [TERMINAL, utility=3]
        12 [TERMINAL, utility=12]
        8 [TERMINAL, utility=8]
    C [MIN]
        2 [TERMINAL, utility=2]
        4 [TERMINAL, utility=4]
        6 [TERMINAL, utility=6]


---
## 4. Minimax

Mathematically, the minimax value is defined recursively by

$$
V(s)=
\begin{cases}
U(s), & s \text{ terminal}\\
\max_{c \in \text{Children}(s)} V(c), & s \text{ is MAX}\\
\min_{c \in \text{Children}(s)} V(c), & s \text{ is MIN}
\end{cases}
$$

This assumes:
- MAX tries to maximize value
- MIN tries to minimize value
- both play optimally

In [20]:
def minimax(node: GameNode) -> float:
    if node.node_type == "TERMINAL":
        return node.utility

    if node.node_type == "MAX":
        return max(minimax(child) for child in node.children)

    if node.node_type == "MIN":
        return min(minimax(child) for child in node.children)

    raise ValueError("Unsupported node type for minimax:", node.node_type)

In [21]:
value_A = minimax(A)
print(f"Minimax value at root A: {value_A}")

Minimax value at root A: 3


---
## 5. Best Action at the Root

Usually we do not just want the value of the root.
We want the action / child that should be chosen.

In [22]:
def minimax_decision(node: GameNode) -> Tuple[str, float]:
    if node.node_type != "MAX":
        raise ValueError("This function assumes the root is a MAX node")

    best_child = None
    best_value = -math.inf

    for child in node.children:
        value = minimax(child)
        print(f"Child {child.name} has minimax value {value}")
        if value > best_value:
            best_value = value
            best_child = child

    return best_child.name, best_value

In [23]:
best_move, best_value = minimax_decision(A)
print("Best move:", best_move)
print("Best value:", best_value)

Child B has minimax value 3
Child C has minimax value 2
Best move: B
Best value: 3


---
## 6. Why Greedy Search Fails

A greedy strategy looks only at the immediate children and chooses the child
that appears best right now.

But in adversarial games, the opponent responds optimally.

So the locally best move may not be globally optimal.

Suppose MAX sees two children:

- Left child has an immediate attractive-looking heuristic
- Right child looks less attractive at first

But after MIN responds:
- Left leads to a terrible outcome
- Right leads to a better guaranteed outcome

Therefore, greedy search is not reliable in adversarial settings.

In [24]:
# Example of a tree where immediate appearance can be deceptive
leaf1 = GameNode("10", "TERMINAL", utility=10)
leaf2 = GameNode("-100", "TERMINAL", utility=-100)
leaf3 = GameNode("4", "TERMINAL", utility=4)
leaf4 = GameNode("5", "TERMINAL", utility=5)

L = GameNode("L", "MIN", children=[leaf1, leaf2])  # MIN makes it -100
R = GameNode("R", "MIN", children=[leaf3, leaf4])  # MIN makes it 4
root_greedy = GameNode("Root", "MAX", children=[L, R])

print("Minimax value of root:", minimax(root_greedy))
print("Best move from root:", minimax_decision(root_greedy))

Minimax value of root: 4
Child L has minimax value -100
Child R has minimax value 4
Best move from root: ('R', 4)


### Reflection
Why is the left branch deceptive?

Because MAX may notice a large leaf (10), but MIN controls that branch and will choose -100 instead.
So the true value of L is min(10, -100) = -100.

---
## 7. Counting Explored Nodes

To compare minimax and alpha-beta, we count how many nodes are visited.

In [25]:
def minimax_count(node: GameNode) -> Tuple[float, int]:
    if node.node_type == "TERMINAL":
        return node.utility, 1

    values = []
    count = 1 # count current node

    for child in node.children:
        child_value, child_count = minimax_count(child)
        values.append(child_value)
        count += child_count

    if node.node_type == "MAX":
        return max(values), count
    elif node.node_type == "MIN":
        return min(values), count
    else:
        raise ValueError("Unsupported node type:", node.node_type)

In [26]:
value, count = minimax_count(A)
print("Minimax value", value)
print("Nodes visited", count)

Minimax value 3
Nodes visited 9


---
## 8. Alpha-Beta Pruning

Alpha-beta pruning improves minimax by avoiding branches that cannot affect the final answer.

Definitions:
- alpha = best value MAX can guarantee so far
- beta = best value MIN can guarantee so far

Pruning conditions:
- at MAX node: if value >= beta, prune
- at MIN node: if value <= alpha, prune

At a MAX node:
- we update
$$
\alpha \leftarrow \max(\alpha, v)
$$
- if
$$
v \ge \beta
$$
then we stop exploring remaining children.

At a MIN node:
- we update
$$
\beta \leftarrow \min(\beta, v)
$$
- if
$$
v \le \alpha
$$
then we stop exploring remaining children.

In [27]:
def alpha_beta(
    node: GameNode,
    alpha: float = -math.inf,
    beta: float = math.inf
) -> float:
    if node.node_type == "TERMINAL":
        return node.utility

    if node.node_type == "MAX":
        value = -math.inf
        for child in node.children:
            value = max(value, alpha_beta(child, alpha, beta))
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return value

    if node.node_type == "MIN":
        value = math.inf
        for child in node.children:
            value = min(value, alpha_beta(child, alpha, beta))
            beta = min(beta, value)
            if beta <= alpha:
                break
        return value

    raise ValueError(f"Unsupported node type for alpha-beta: {node.node_type}")

In [28]:
print("Alpha-beta value at root:", alpha_beta(A))

Alpha-beta value at root: 3


---
## 9. Alpha-Beta with Trace

To really understand pruning, let us print what happens.

In [29]:
def alpha_beta_trace(
    node: GameNode,
    alpha: float = -math.inf,
    beta: float = math.inf,
    depth: int = 0,
) -> float:
    indent = "    " * depth

    if node.node_type == "TERMINAL":
        print(f"{indent}Visit termianl {node.name}: {node.utility}")
        return node.utility

    print(f"{indent}Entering {node.name} ({node.node_type}) with alpha={alpha}, beta={beta}")

    if node.node_type == "MAX":
        value = -math.inf
        for child in node.children:
            child_value = alpha_beta_trace(child, alpha, beta, depth+1)
            value = max(value, child_value)
            alpha = max(alpha, value)
            print(f"{indent}Updated MAX node {node.name}: value={value}, alpha={alpha}, beta={beta}")
            if alpha >= beta:
                print(f"{indent}Prune remaining children of {node.name} because alpha >= beta")
                break
        return value

    if node.node_type == "MIN":
        value = math.inf
        for child in node.children:
            child_value = alpha_beta_trace(child, alpha, beta, depth+1)
            value = min(value, child_value)
            beta = min(beta, value)
            print(f"{indent}Updated MIN node {node.name}: value={value}, alpha={alpha}, beta={beta}")
            if beta <= alpha:
                print(f"{indent}Prune remaining children of {node.name} because beta <= alpha")
                break
        return value
    raise ValueError(f"Unsupported node type: {node.node_type}")

In [30]:
alpha_beta_trace(A)

Entering A (MAX) with alpha=-inf, beta=inf
    Entering B (MIN) with alpha=-inf, beta=inf
        Visit termianl 3: 3
    Updated MIN node B: value=3, alpha=-inf, beta=3
        Visit termianl 12: 12
    Updated MIN node B: value=3, alpha=-inf, beta=3
        Visit termianl 8: 8
    Updated MIN node B: value=3, alpha=-inf, beta=3
Updated MAX node A: value=3, alpha=3, beta=inf
    Entering C (MIN) with alpha=3, beta=inf
        Visit termianl 2: 2
    Updated MIN node C: value=2, alpha=3, beta=2
    Prune remaining children of C because beta <= alpha
Updated MAX node A: value=3, alpha=3, beta=inf


3

---
## 10. A Tree Where Pruning Is Easier to Observe

In [31]:
# Leaves
t3 = GameNode("3", "TERMINAL", utility=3)
t5 = GameNode("5", "TERMINAL", utility=5)
t6 = GameNode("6", "TERMINAL", utility=6)
t9 = GameNode("9", "TERMINAL", utility=9)
t1 = GameNode("1", "TERMINAL", utility=1)
t2 = GameNode("2", "TERMINAL", utility=2)

# Internal MIN nodes
M1 = GameNode("M1", "MIN", children=[t3, t5])   # value = 3
M2 = GameNode("M2", "MIN", children=[t6, t9])   # value = 6
M3 = GameNode("M3", "MIN", children=[t1, t2])   # value = 1

# Root MAX
R = GameNode("R", "MAX", children=[M1, M2, M3])

print("Minimax value:", minimax(R))
print("Alpha-beta value:", alpha_beta(R))

Minimax value: 6
Alpha-beta value: 6


In [32]:
alpha_beta_trace(R)

Entering R (MAX) with alpha=-inf, beta=inf
    Entering M1 (MIN) with alpha=-inf, beta=inf
        Visit termianl 3: 3
    Updated MIN node M1: value=3, alpha=-inf, beta=3
        Visit termianl 5: 5
    Updated MIN node M1: value=3, alpha=-inf, beta=3
Updated MAX node R: value=3, alpha=3, beta=inf
    Entering M2 (MIN) with alpha=3, beta=inf
        Visit termianl 6: 6
    Updated MIN node M2: value=6, alpha=3, beta=6
        Visit termianl 9: 9
    Updated MIN node M2: value=6, alpha=3, beta=6
Updated MAX node R: value=6, alpha=6, beta=inf
    Entering M3 (MIN) with alpha=6, beta=inf
        Visit termianl 1: 1
    Updated MIN node M3: value=1, alpha=6, beta=1
    Prune remaining children of M3 because beta <= alpha
Updated MAX node R: value=6, alpha=6, beta=inf


6

### Reflection
Observe where pruning occurred.
Why was it safe to prune?
Which bound caused the cutoff?

---
## 11. Comparing Node Counts

In [33]:
def alpha_beta_count(
    node: GameNode,
    alpha: float = -math.inf,
    beta: float = math.inf
) -> Tuple[float, int]:
    if node.node_type == "TERMINAL":
        return node.utility, 1

    count = 1

    if node.node_type == "MAX":
        value = -math.inf
        for child in node.children:
            child_value, child_count = alpha_beta_count(child, alpha, beta)
            count += child_count
            value = max(value, child_value)
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return value, count

    if node.node_type == "MIN":
        value = math.inf
        for child in node.children:
            child_value, child_count = alpha_beta_count(child, alpha, beta)
            count += child_count
            value = min(value, child_value)
            beta = min(beta, value)
            if beta <= alpha:
                break
        return value, count

    raise ValueError(f"Unsupported node type: {node.node_type}")

In [34]:
minimax_value, minimax_nodes = minimax_count(R)
ab_value, ab_nodes = alpha_beta_count(R)

print("Minimax value:", minimax_value, "| nodes:", minimax_nodes)
print("Alpha-beta value:", ab_value, "| nodes:", ab_nodes)

Minimax value: 6 | nodes: 10
Alpha-beta value: 6 | nodes: 9


Alpha-beta should return the same value as minimax, but usually by visiting fewer nodes.

---
## 12. Depth-Limited Search

In realistic games, full search is usually impossible because the tree is too large.

So instead:
- search only to depth limit d
- if depth limit is reached before terminal state, use an evaluation function

This gives an approximation to minimax.

In [35]:
@dataclass
class EvalNode:
    name: str
    node_type: str # "MAX", "MIN", "TERMINAL"
    children: List["EvalNode"] = field(default_factory=list)
    utility: Optional[float] = None
    heuristic: Optional[float] = None

In [36]:
def depth_limited_minimax(node: EvalNode, depth_limit: int) -> float:
    if node.node_type == "TERMINAL":
        return node.utility

    if depth_limit == 0:
        if node.heuristic is None:
            raise ValueError(f"Node {node.name} needs a heuristic value at cutoff")
        return node.heuristic

    if node.node_type == "MAX":
        return max(depth_limited_minimax(child, depth_limit-1) for child in node.children)

    if node.node_type == "MIN":
        return min(depth_limited_minimax(child, depth_limit-1) for child in node.children)

    raise ValueError(f"Unsupported node type: {node.node_type}")

---
## 13. Example Tree with Heuristic Values
Suppose some internal nodes are not terminal, but we stop early and estimate them.

In [37]:
# Depth-2 internal nodes with heuristic values
h1 = EvalNode("H1", "MAX", heuristic=7)
h2 = EvalNode("H2", "MAX", heuristic=3)
h3 = EvalNode("H3", "MAX", heuristic=6)
h4 = EvalNode("H4", "MAX", heuristic=2)

m1 = EvalNode("M1", "MIN", children=[h1, h2])
m2 = EvalNode("M2", "MIN", children=[h3, h4])

root_eval = EvalNode("Root", "MAX", children=[m1, m2])

In [38]:
print("Depth-limited minimax value:", depth_limited_minimax(root_eval, depth_limit=2))

Depth-limited minimax value: 3


- `M1 = min(7, 3) = 3`
- `M2 = min(6, 2) = 2`
- `Root = max(3, 2) = 3`

---
## 14. Evaluation Functions

A common form is a weighted linear function:

$$
\text{Eval}(s)=\sum_{i=1}^{n} w_i f_i(s)
$$

where:
- $f_i(s)$ are features
- $w_i$ are weights

In [39]:
def linear_evaluation(
    features: Dict[str, float],
    weights: Dict[str, float]
) -> float:
    score = 0.0
    for key, value in features.items():
        score += weights.get(key, 0.0) * value
    return score

In [40]:
# Toy chess-like features
features = {
    "material": 2,
    "mobility": 5,
    "king_safety": -1
}

weights = {
    "material": 10,
    "mobility": 1.5,
    "king_safety": 4
}

score = linear_evaluation(features, weights)
print("Evaluation score:", score)

Evaluation score: 23.5


### Reflection
What does a large positive score mean?
What happens if the weights are badly chosen?
Why is evaluation difficult in real games?

---
## 15. Horizon Effect

The horizon effect occurs when a dangerous event lies just beyond the search depth.

The search stops too early and fails to notice what is coming.

This is one reason why depth-limited search can make bad decisions.

---
## 16. Expectiminimax

Minimax assumes no randomness.
But some games include chance events, such as dice rolls.

For chance nodes, we use expected value:

$$
V(s)=\sum_i p_i V(s_i)
$$

In [41]:
def expectiminimax(node: GameNode) -> float:
    if node.node_type == "TERMINAL":
        return node.utility

    if node.node_type == "MAX":
        return max(expectiminimax(child) for child in node.children)

    if node.node_type == "MIN":
        return min(expectiminimax(child) for child in node.children)

    if node.node_type == "CHANCE":
        if node.probabilities is None:
            raise ValueError(f"Chance node {node.name} must have probabilities")
        if len(node.probabilities) != len(node.children):
            raise ValueError(f"Probabilities and children mismatch at {node.name}")

        return sum(p * expectiminimax(child) for p, child in zip(node.probabilities, node.children))

    raise ValueError(f"Unsupported node type: {node.node_type}")

---
## 17. Example with a Chance Node

Suppose MAX can choose between:

- Left: deterministic guaranteed utility 3
- Right: a chance node with:
  - probability 0.5 -> utility 2
  - probability 0.5 -> utility 8

Expected value of Right:
$$
0.5 \cdot 2 + 0.5 \cdot 8 = 5
$$

So MAX should choose Right.

In [42]:
left_terminal = GameNode("LeftTerminal", "TERMINAL", utility=3)

chance_leaf1 = GameNode("ChanceLeaf1", "TERMINAL", utility=2)
chance_leaf2 = GameNode("ChanceLeaf2", "TERMINAL", utility=8)
chance_node = GameNode("ChanceNode", "CHANCE", children=[chance_leaf1, chance_leaf2], probabilities=[0.5, 0.5])

root_chance = GameNode("RootChance", "MAX", children=[left_terminal, chance_node])

print("Expectiminimax value at root:", expectiminimax(root_chance))

Expectiminimax value at root: 5.0


- Left = 3
- Right = 0.5(2) + 0.5(8) = 5
- Root = max(3, 5) = 5

In [43]:
def expectiminimax_decision(node: GameNode) -> Tuple[str, float]:
    if node.node_type != "MAX":
        raise ValueError("Root must be MAX for this decision function.")

    best_child = None
    best_value = -math.inf

    for child in node.children:
        value = expectiminimax(child)
        print(f"Child {child.name} has expectiminimax value {value}")
        if value > best_value:
            best_value = value
            best_child = child

    return best_child.name, best_value

In [44]:
best_move, best_value = expectiminimax_decision(root_chance)
print("Best move:", best_move)
print("Best value:", best_value)

Child LeftTerminal has expectiminimax value 3
Child ChanceNode has expectiminimax value 5.0
Best move: ChanceNode
Best value: 5.0


---
## 18. Summary Comparison

### Minimax
- deterministic
- two-player adversarial
- assumes optimal opponent
- uses max / min backups

### Alpha-Beta
- same result as minimax
- prunes branches that cannot matter
- usually more efficient

### Depth-Limited Search
- practical approximation
- stops at fixed depth
- uses evaluation function at cutoff

### Expectiminimax
- used when chance/randomness exists
- adds expectation at chance nodes

---
## 19. Complexity Discussion

If branching factor is $b$ and depth is $d$:

### Minimax
$$
O(b^d)
$$

### Alpha-Beta
- worst case:
$$
O(b^d)
$$
- best case with good ordering:
$$
O(b^{d/2})
$$

This is why alpha-beta pruning is so important in practice.